## Enter your NC file path in the following code and then just call the function to generate wind files.

In [13]:
#If you have not installed the xarray, just remove # from the below line
# !pip install xarray
import xarray as xr

# 1. Open your NetCDF
file_path=('file_path.nc')
ds = xr.open_dataset(file_path)

#2. extract Variables
u_10 = ds["u10"]
v_10 = ds["v10"]
times = ds["valid_time"].values
lon=ds['longitude'].values
lat=ds['latitude'].values

#3. some information for the header rows
num_rows = u_10.shape[1]
num_cols = u_10.shape[2]

#5. define a function just for the headlines
def headline(txt_file,direction:'str'):
        txt_file.write("FileVersion = 1.03\n") # do not change this value
        txt_file.write("Filetype = meteo_on_equidistant_grid\n") # do not change this value
        txt_file.write("NODATA_value = -999.000\n") # do not change this value
        txt_file.write(f"n_cols = {num_cols}\n") # number of columns of data for each time_step
        txt_file.write(f"n_rows = {num_rows}\n") # number of rows of data for each time_step
        txt_file.write("grid_unit = degree\n")   # do not change this value
        txt_file.write(f"x_llcorner = {lon.min():.2f}\n")
        txt_file.write(f"y_llcorner = {lat.min():.2f}\n")
        txt_file.write("dx = 0.25\n") #This is the grid space for ERA5 data
        txt_file.write("dy = 0.25\n") #This is the grid space for ERA5 data
        txt_file.write("n_quantity = 1\n")  # do not change this value
        txt_file.write(f"quantity1 = {direction}_wind\n") # change to y_wind if you want to generate y-dir wind file 
        txt_file.write("unit1 = m s-1\n") # do not change this value 
        return
# 6. define the 
def wind(N_hours,start_date:'str',start_time:'str'='00:00:00',time_zone:'str'='+00:00'):
    """
    returns wind files in x and y directions.
    Parameters:
    N_hours(int): How many hours of wind data needed for simulation.
    start_date(str): reference date for wind forced wave simulation. format= 'YYYY-MM-'DD'
    start_time(str): reference time . format= 'HH:MM:SS'
    time_zone:time_zone_difference with GMT. data is already matched with GMT. 
    """

# 7. Write file with explicit newline formatting (\n)
    with open("xwind.wnd", "w", newline="\n") as x_file, \
         open('ywind.wnd', 'w', newline='\n') as y_file:

        # --- SWAN SPECIFIC METEO HEADER BLOCK ---
        headline(x_file,'x')
        headline(y_file,'y')
        # --- DATA LOOP ---
        for i in range(N_hours):
            # first let's check if the time actually exist in The NC file
            add_time = np.timedelta64(60*i, 'm')
            base_date = np.datetime64(f"{start_date}T{start_time}")
            target_date= base_date + add_time
            j = np.where(times == target_date)[0] 
            if j.size==0:
                raise ValueError(f"""Time Error: The time {target_date} does not exist in NC file...
                ... but {i} time steps are printed out in your wnd file""")
                pass
            # Write the time row
            x_file.write(
                f"Time = {i * 1.0:.1f} hours since {start_date} {start_time} {time_zone}\n")  
            grid_2d = u_10[j, :, :].values
            # Write the values row
            np.savetxt(x_file, grid_2d.squeeze(), fmt='%.2f', delimiter=' ')
            y_file.write(
                f"Time = {i * 1.0:.1f} hours since {start_date} {start_time} {time_zone}\n")  
            grid_2d = v_10[j, :, :].values
            np.savetxt(y_file, grid_2d.squeeze(), fmt='%.2f', delimiter=' ')
    return

In [ ]:
wind(50,'2019-02-01')

## If You Have a GEBCO NC file, replace your NC file path with the 'path.nc' and run the following code. 

In [3]:
#importing sample file (GEBCO)
import pandas as pd
import xarray as xr
GEBCO_NC_path='path.nc'
depth=xr.open_dataset(GEBCO_NC_path)
Lon_mesh,Lat_mesh=np.meshgrid(depth['lon'].values,depth['lat'].values)
flat_lon=Lon_mesh.flatten()
flat_lat=Lat_mesh.flatten()
flat_depth=depth['elevation'].values.flatten()
# DF_CS=pd.read_fwf('./Caspian_Sea.txt',header=None)
# DF_CS.head()
# DF_CG.drop(columns=[2,3],inplace=True)
# target_lon, target_lat = transformer.transform(DF_CG[0], DF_CG[1])
# target_lon

In [4]:
DF_depth=pd.DataFrame([flat_lon,flat_lat,flat_depth])
DF_depth=DF_depth.T
DF_depth.columns=['lon','lat','Z']
#To use the file in Delft3d, depth values must be in positive
DF_depth['Z']=-1*(DF_depth['Z'])
DF_depth.to_excel('./GEBCO_sample_file.xlsx',index=False,header=False) #Generate an excel file and just copy values from excel to a xyz file

## If you have hydrographic survey UTM file, use the following code and replace your file path with 'file.txt' 

In [ ]:
# If you have not installed the pyproj, just remove # from the below line
#!pip install pyproj
from pyproj import Transformer
import pandas as pd
UTM_file_path='./file.txt'
DF_UTM=pd.read_fwf(UTM_file_path,header=None)
DF_UTM.columns=['x','y','z']
transformer = Transformer.from_crs("EPSG:32639", "EPSG:4326", always_xy=True) #Enter your UTM zone instead of 32639. 
lon_transformed, lat_transformed = Transform.transform(DF_UTM['x'].values, DF_UTM['y'].values)
DF_lonlat = pd.DataFrame({
    'lon': lon_transformed,
    'lat': lat_transformed,
    'z': DF_UTM['z'].values})
DF_lonlat.to_excel('./lonlat_sample_file.xlsx',header=None,index=None)